# CS431 — Depth Anything 3 Exposure-Robust Reference Selection (v2)

## Cải tiến so với bản chạy pilot v1
1. **Sửa lỗi `F1 = 0.0` trong `cs431/geometry.py`**:
   - Thay căn chỉnh Umeyama chỉ dùng 4 tâm camera (dễ bị bất định xoay quanh cung camera) bằng căn chỉnh **Sim(3) kết hợp cả ma trận xoay `R_i` và tâm camera `t_i`** (`align_poses_sim3`).
   - Trích xuất thêm `sparse/0` của COLMAP và chạy bước **chẩn đoán hệ tọa độ** (`transforms_train.json` vs `sparse/0` với `colmap2world.txt` và `cull_pointcloud.ply`) trước khi chạy toàn bộ thí nghiệm.
2. **Cải tiến bộ chọn ảnh tham chiếu chỉ dùng RGB (`cs431/reference.py`)**:
   - Chuẩn hóa gamma thích nghi + CLAHE + tăng ngân sách SIFT (`nfeatures=2000, contrastThreshold=0.02, ratio=0.80, min_pairs=8`) để tránh suy biến `overlap = 0` trên ảnh thiếu sáng.
   - Kết hợp độ phơi sáng bão hòa mềm (`tanh(exposure / 0.15)`) với **độ trung tâm phơi sáng (`bridge`)** theo trung vị log-độ sáng, giúp ảnh tham chiếu làm cầu nối giữa ảnh tối và ảnh sáng thay vì chọn cực sáng lạc lõng.
   - Chuẩn hóa min-max hai nhánh trước khi cộng (`combined = 0.5 * norm(exposure_bridge) + 0.5 * norm(overlap)`).
3. **Tối ưu tốc độ chạy**: tái sử dụng checkpoint DA3-Base và `gt_points` đã lấy mẫu voxel cho cả 3 điều kiện trên mỗi cảnh.


In [ ]:
import csv
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
from collections import defaultdict
from pathlib import Path
from time import perf_counter

import numpy as np
from PIL import Image

WORK_DIR = Path('/kaggle/working')
OUT_DIR = WORK_DIR / 'outputs'
TMP_DIR = Path('/tmp/cs431_run')
REPO_DIR = TMP_DIR / 'Depth-Anything-3'
DATA_DIR = TMP_DIR / 'realx3d'
PC_DIR = TMP_DIR / 'pointclouds'

DA3_REPO_URL = 'https://github.com/ByteDance-Seed/Depth-Anything-3.git'
DA3_COMMIT = '3d835ec1a5802d64a8b8b15f817a1ab54809bfe4'
HF_REALX3D_BASE = 'https://huggingface.co/datasets/ToferFish/RealX3D/resolve/main'
VARYEXP_TAR_URL = f'{HF_REALX3D_BASE}/data_4/data4_varyexp.tar.gz'

PILOT_SCENES = ['Ujikintoki', 'BlueHawaii', 'Chocolate', 'Cupcake']
NUM_VIEWS = 4
RESOLUTION = 504

for d in (OUT_DIR, TMP_DIR, DATA_DIR, PC_DIR):
    d.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=None, env=None, check=True):
    print(f"$ {' '.join(map(str, cmd))}", flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

def locate_cs431_code() -> Path:
    for found in Path('/kaggle/input').rglob('run_experiment.py'):
        return found.parent
    raise FileNotFoundError('Could not locate cs431/run_experiment.py under /kaggle/input')

run_cmd(['nvidia-smi'], check=False)
if not REPO_DIR.exists():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    run_cmd(['git', 'init'], cwd=REPO_DIR)
    run_cmd(['git', 'remote', 'add', 'origin', DA3_REPO_URL], cwd=REPO_DIR)
    run_cmd(['git', 'fetch', '--depth', '1', 'origin', DA3_COMMIT], cwd=REPO_DIR)
    run_cmd(['git', 'checkout', 'FETCH_HEAD'], cwd=REPO_DIR)

cs431_src = locate_cs431_code()
cs431_dst = REPO_DIR / 'cs431'
if cs431_dst.exists():
    shutil.rmtree(cs431_dst)
shutil.copytree(cs431_src, cs431_dst)
print(f'Copied cs431 v2 from {cs431_src} -> {cs431_dst}', flush=True)

pkgs = [
    'addict', 'omegaconf', 'einops>=0.8', 'huggingface_hub', 'safetensors',
    'evo', 'plyfile', 'trimesh', 'e3nn', 'moviepy==1.0.3', 'imageio',
    'pycolmap', 'pillow_heif'
]
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

os.environ['HF_HOME'] = str(TMP_DIR / 'hf_cache')
for p in (str(REPO_DIR / 'src'), str(REPO_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

env = os.environ.copy()
env['PYTHONPATH'] = f"{REPO_DIR / 'src'}:{REPO_DIR}"
run_cmd([sys.executable, '-m', 'unittest', 'cs431.test_experiment', '-v'], cwd=REPO_DIR, env=env)


## Bước 2: Stream dữ liệu 4 cảnh pilot từ RealX3D (kèm `sparse/0`)
Stream trực tiếp `data4_varyexp.tar.gz` mà không lưu file nén 13.4 GB xuống ổ đĩa, chỉ trích xuất `transforms_train.json`, `colmap2world.txt`, `sparse/0/*` và ảnh `train*` của 4 cảnh pilot.


In [ ]:
def stream_extract_varyexp(target_scenes: list[str]):
    target_set = set(target_scenes)
    extracted_files = 0
    extracted_bytes = 0
    t0 = perf_counter()
    print(f'Streaming {VARYEXP_TAR_URL} for scenes: {target_scenes}', flush=True)
    proc = subprocess.Popen(
        ['curl', '-fsSL', '--retry', '5', '--retry-delay', '5', VARYEXP_TAR_URL],
        stdout=subprocess.PIPE,
    )
    try:
        with tarfile.open(fileobj=proc.stdout, mode='r|gz') as tar:
            for member in tar:
                if not member.isfile():
                    continue
                parts = Path(member.name).parts
                if len(parts) < 3:
                    continue
                scene = parts[1] if parts[0] == 'varyexp' else parts[0]
                rel_parts = parts[2:] if parts[0] == 'varyexp' else parts[1:]
                if scene not in target_set or not rel_parts:
                    continue
                top = rel_parts[0]
                is_meta = top in ('transforms_train.json', 'colmap2world.txt')
                is_sparse = top == 'sparse'
                is_train_img = top.startswith('train') and member.name.lower().endswith(('.jpg', '.jpeg', '.png'))
                if not (is_meta or is_sparse or is_train_img):
                    continue
                dst = DATA_DIR / scene / Path(*rel_parts)
                dst.parent.mkdir(parents=True, exist_ok=True)
                src_f = tar.extractfile(member)
                if src_f is not None:
                    with dst.open('wb') as out_f:
                        shutil.copyfileobj(src_f, out_f)
                    extracted_files += 1
                    extracted_bytes += member.size
                    if extracted_files % 100 == 0:
                        print(f'  extracted {extracted_files} files ({extracted_bytes / 1e9:.2f} GB) in {perf_counter() - t0:.1f}s', flush=True)
    finally:
        if proc.stdout:
            proc.stdout.close()
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'curl exited with status {ret}')
    print(f'Done streaming varyexp: {extracted_files} files ({extracted_bytes / 1e9:.2f} GB) in {perf_counter() - t0:.1f}s', flush=True)

def download_pointcloud(scene: str) -> tuple[Path, Path | None]:
    url = f'{HF_REALX3D_BASE}/pointclouds/{scene}.tar.gz'
    scene_pc_dir = PC_DIR / scene
    scene_pc_dir.mkdir(parents=True, exist_ok=True)
    ply_path = c2w_npy = None
    print(f'Streaming pointcloud {url}', flush=True)
    proc = subprocess.Popen(
        ['curl', '-fsSL', '--retry', '5', '--retry-delay', '5', url],
        stdout=subprocess.PIPE,
    )
    try:
        with tarfile.open(fileobj=proc.stdout, mode='r|gz') as tar:
            for member in tar:
                if not member.isfile():
                    continue
                name = Path(member.name).name
                if name in ('cull_pointcloud.ply', 'colmap2world.npy'):
                    dst = scene_pc_dir / name
                    src_f = tar.extractfile(member)
                    if src_f is not None:
                        with dst.open('wb') as out_f:
                            shutil.copyfileobj(src_f, out_f)
                    if name == 'cull_pointcloud.ply':
                        ply_path = dst
                    elif name == 'colmap2world.npy':
                        c2w_npy = dst
    finally:
        if proc.stdout:
            proc.stdout.close()
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'curl failed for {url} (exit {ret})')
    if ply_path is None:
        raise FileNotFoundError(f'cull_pointcloud.ply not found in {url}')
    return ply_path, c2w_npy

stream_extract_varyexp(PILOT_SCENES)


## Bước 3: Chẩn đoán hệ tọa độ trên cảnh đầu tiên (`Ujikintoki`) & tải mô hình DA3-Base
So sánh trực tiếp vị trí camera và khoảng cách đám mây điểm giữa:
- `transforms_train.json` + `colmap2world.txt`
- `sparse/0` (COLMAP gốc) + `colmap2world.txt`
- `sparse/0` + `inv(colmap2world.txt)`

Chọn đúng chế độ tọa độ khớp với `cull_pointcloud.ply` trước khi chạy toàn bộ thí nghiệm.


In [ ]:
import torch
from plyfile import PlyData
from depth_anything_3.api import DepthAnything3
from cs431.geometry import cloud_metrics, predicted_poses, relative_pose_metrics, to_metric_poses, voxel_sample
from cs431.run_experiment import DA3_BASE, DA3_BASE_REVISION, evaluate_views, load_colmap_views, load_views

device = torch.device('cuda:0')
assert torch.cuda.is_available(), 'CUDA is required'
print('Loading DA3-Base on', torch.cuda.get_device_name(0), flush=True)
model = DepthAnything3.from_pretrained(DA3_BASE, revision=DA3_BASE_REVISION).to(device).eval()

def load_gt_cloud(ply_path: Path) -> np.ndarray:
    vertices = PlyData.read(str(ply_path))['vertex'].data
    xyz = np.column_stack([vertices[a] for a in 'xyz']).astype(float)
    return voxel_sample(xyz)

# Diagnostic on Ujikintoki
diag_scene = PILOT_SCENES[0]
diag_dir = DATA_DIR / diag_scene
diag_ply, diag_c2w_npy = download_pointcloud(diag_scene)
diag_gt_pts = load_gt_cloud(diag_ply)
c2w_mat = np.loadtxt(diag_dir / 'colmap2world.txt')
print(f'[{diag_scene}] GT cloud voxel count={len(diag_gt_pts)}, min={diag_gt_pts.min(0).round(3)}, max={diag_gt_pts.max(0).round(3)}, mean={diag_gt_pts.mean(0).round(3)}')
print('colmap2world.txt:\n', np.round(c2w_mat, 4))

diag_paths, json_poses = load_views(diag_dir / 'transforms_train.json', diag_dir, NUM_VIEWS)
diag_imgs = [Image.open(p).convert('RGB') for p in diag_paths]
diag_pred = model.inference(diag_imgs, ref_view_strategy='saddle_balanced', process_res=RESOLUTION)

candidates = {}
candidates['json + c2w'] = to_metric_poses(json_poses, c2w_mat)
candidates['json + inv(c2w)'] = to_metric_poses(json_poses, np.linalg.inv(c2w_mat))
sparse0 = diag_dir / 'sparse' / '0'
if sparse0.is_dir():
    _, colmap_poses = load_colmap_views(sparse0, diag_paths)
    candidates['colmap_sparse + c2w'] = to_metric_poses(colmap_poses, c2w_mat)
    candidates['colmap_sparse + inv(c2w)'] = to_metric_poses(colmap_poses, np.linalg.inv(c2w_mat))

best_mode = None
best_f1 = -1.0
for mode_name, metric_poses in candidates.items():
    m = cloud_metrics(diag_pred, metric_poses, diag_gt_pts)
    rel = relative_pose_metrics(predicted_poses(diag_pred.extrinsics), metric_poses)
    cam_mean = metric_poses[:, :3, 3].mean(0).round(3)
    print(f'  mode={mode_name:24s} | cam_mean={cam_mean} | f1={m.get("f1"):.4f} | f1_10cm={m.get("f1_10cm"):.4f} | med_p2g={m.get("median_pred_to_gt_m"):.4f}m | auc10={rel["auc10"]:.4f}')
    if m.get('f1', -1) > best_f1:
        best_f1 = m['f1']
        best_mode = mode_name

print(f'>>> Selected verified coordinate mode: {best_mode} (F1={best_f1:.4f})', flush=True)


## Bước 4: Chạy thực nghiệm v2 trên 4 cảnh pilot × 3 điều kiện (`--sweep`)
Đánh giá 5 cấu hình: `saddle_balanced` (DA3 mặc định), `exposure`, `exposure_bridge`, `overlap`, `combined`, cùng lượt quét đầy đủ `sweep` và `oracle_f1`.


In [ ]:
def _exposure_sort_key(folder_name: str):
    m = re.search(r'(\d+)', folder_name)
    return (0, int(m.group(1))) if m else (1, folder_name)

def build_scene_conditions(scene_dir: Path, num_views: int = NUM_VIEWS):
    transforms_path = scene_dir / 'transforms_train.json'
    frames = json.loads(transforms_path.read_text(encoding='utf-8'))['frames']
    idxs = np.linspace(0, len(frames) - 1, num_views, dtype=int)
    sampled_frames = [frames[i] for i in idxs]
    view_names = [Path(f['file_path']).name for f in sampled_frames]
    train_dirs = sorted(
        [p for p in scene_dir.iterdir() if p.is_dir() and p.name.startswith('train')],
        key=lambda p: _exposure_sort_key(p.name),
    )
    iso_dirs = [p for p in train_dirs if re.search(r'\d+', p.name)]
    conditions = {}
    default_paths = [(scene_dir / f['file_path']) for f in sampled_frames]
    if all(p.is_file() for p in default_paths):
        conditions['default_json'] = default_paths
    complete_iso = [d for d in iso_dirs if all((d / name).is_file() for name in view_names)]
    if len(complete_iso) >= 2:
        pick_indices = np.linspace(0, len(complete_iso) - 1, num_views, dtype=int)
        conditions['varyexp_mixed'] = [complete_iso[pi] / name for pi, name in zip(pick_indices, view_names)]
        mid_dir = complete_iso[len(complete_iso) // 2]
        conditions['uniform_mid'] = [mid_dir / name for name in view_names]
    return transforms_path, conditions

def load_verified_poses(scene_dir: Path, transforms_path: Path, img_paths: list[Path], mode: str):
    c2w_mat = np.loadtxt(scene_dir / 'colmap2world.txt')
    if mode.startswith('colmap_sparse'):
        paths, poses = load_colmap_views(scene_dir / 'sparse' / '0', img_paths)
    else:
        paths, poses = load_views(transforms_path, scene_dir, len(img_paths), img_paths)
    mat = np.linalg.inv(c2w_mat) if 'inv(c2w)' in mode else c2w_mat
    metric_poses = to_metric_poses(poses, mat)
    return paths, poses, metric_poses

summary_rows = []
cached_ply = {diag_scene: (diag_ply, diag_gt_pts)}

for scene in PILOT_SCENES:
    scene_dir = DATA_DIR / scene
    if scene in cached_ply:
        ply_path, gt_points = cached_ply[scene]
    else:
        ply_path, _ = download_pointcloud(scene)
        gt_points = load_gt_cloud(ply_path)
    transforms_path, conditions = build_scene_conditions(scene_dir, NUM_VIEWS)

    for cond_name, img_paths in conditions.items():
        paths, gt_poses, gt_metric_poses = load_verified_poses(scene_dir, transforms_path, img_paths, best_mode)
        images = [Image.open(p).convert('RGB') for p in paths]

        def infer(ordered, strategy, order):
            torch.cuda.synchronize(device)
            torch.cuda.reset_peak_memory_stats(device)
            t0 = perf_counter()
            prediction = model.inference(ordered, ref_view_strategy=strategy, process_res=RESOLUTION)
            torch.cuda.synchronize(device)
            res = {
                'inference_seconds': perf_counter() - t0,
                'peak_cuda_mib': torch.cuda.max_memory_allocated(device) / 1024**2,
            }
            t1 = perf_counter()
            res.update(relative_pose_metrics(predicted_poses(prediction.extrinsics), gt_poses[order]))
            res.update(cloud_metrics(prediction, gt_metric_poses[order], gt_points))
            res['metric_seconds'] = perf_counter() - t1
            return res

        report = {
            'model': DA3_BASE,
            'model_revision': DA3_BASE_REVISION,
            'coordinate_mode': best_mode,
            'device': str(device),
            'resolution': RESOLUTION,
            'images': [str(p) for p in paths],
            **evaluate_views(images, infer, sweep=True),
        }
        out_json = OUT_DIR / f'{scene}_{cond_name}_v2.json'
        out_json.write_text(json.dumps(report, indent=2, allow_nan=False) + '\n', encoding='utf-8')

        sweep_f1s = [r['f1'] for r in report['sweep'] if r.get('f1') is not None]
        sweep_aucs = [r['auc10'] for r in report['sweep']]
        row = {
            'scene': scene,
            'condition': cond_name,
            'selection_s': round(report['selection_seconds'], 4),
        }
        for m in ('saddle_balanced', 'exposure', 'exposure_bridge', 'overlap', 'combined'):
            r = report['runs'][m]
            row[f'{m}_ref'] = r.get('reference_index')
            row[f'{m}_f1'] = round(r['f1'], 4) if r.get('f1') is not None else None
            row[f'{m}_auc10'] = round(r['auc10'], 4)
            row[f'{m}_med_m'] = round(r['median_pred_to_gt_m'], 4) if r.get('median_pred_to_gt_m') is not None else None
        oracle = report.get('oracle_f1')
        row['oracle_f1'] = round(oracle['f1'], 4) if oracle else None
        row['span_f1'] = round(max(sweep_f1s) - min(sweep_f1s), 4) if sweep_f1s else None
        row['oracle_auc10'] = round(max(sweep_aucs), 4)
        row['span_auc10'] = round(max(sweep_aucs) - min(sweep_aucs), 4)
        summary_rows.append(row)
        print(
            f"{scene:12s} {cond_name:14s} | F1: def={row['saddle_balanced_f1']} comb={row['combined_f1']} brg={row['exposure_bridge_f1']} ovl={row['overlap_f1']} orc={row['oracle_f1']} | "
            f"AUC10: def={row['saddle_balanced_auc10']} comb={row['combined_auc10']} orc={row['oracle_auc10']}",
            flush=True,
        )

    shutil.rmtree(PC_DIR / scene, ignore_errors=True)


## Bước 5: Tổng hợp bảng kết quả v2 theo từng cảnh và trung bình theo điều kiện


In [ ]:
csv_path = OUT_DIR / 'summary_v2.csv'
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

md = [
    f'# CS431 Experiment Results v2 (coordinate_mode={best_mode})',
    '',
    '## Per-scene results',
    '| Scene | Condition | Default F1 | Exposure F1 | Bridge F1 | Overlap F1 | Combined F1 | Oracle F1 | Span F1 | Default AUC10 | Combined AUC10 |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for r in summary_rows:
    md.append(
        f"| {r['scene']} | {r['condition']} | {r['saddle_balanced_f1']} | {r['exposure_f1']} | "
        f"{r['exposure_bridge_f1']} | {r['overlap_f1']} | {r['combined_f1']} | {r['oracle_f1']} | "
        f"{r['span_f1']} | {r['saddle_balanced_auc10']} | {r['combined_auc10']} |"
    )

md += ['', '## Mean by condition', '| Condition | Default F1 | Exposure F1 | Bridge F1 | Overlap F1 | Combined F1 | Δ(Comb-Def) F1 | Oracle F1 | Span F1 | Default AUC10 | Combined AUC10 | Δ(Comb-Def) AUC10 |', '| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |']
for cond in ('default_json', 'varyexp_mixed', 'uniform_mid'):
    sub = [r for r in summary_rows if r['condition'] == cond]
    if not sub:
        continue
    def mean_k(k):
        vals = [r[k] for r in sub if r[k] is not None]
        return round(float(np.mean(vals)), 4) if vals else None
    df1, cf1 = mean_k('saddle_balanced_f1'), mean_k('combined_f1')
    da, ca = mean_k('saddle_balanced_auc10'), mean_k('combined_auc10')
    md.append(
        f"| {cond} | {df1} | {mean_k('exposure_f1')} | {mean_k('exposure_bridge_f1')} | "
        f"{mean_k('overlap_f1')} | {cf1} | {round(cf1 - df1, 4) if df1 is not None and cf1 is not None else None} | "
        f"{mean_k('oracle_f1')} | {mean_k('span_f1')} | {da} | {ca} | {round(ca - da, 4)} |"
    )

md_text = '\n'.join(md) + '\n'
(OUT_DIR / 'summary_v2.md').write_text(md_text, encoding='utf-8')
print(md_text)
